In [1]:
%pip install pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 34.6 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 41.9 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
import numpy as np

file_id = '12FsR70V30i08i5uAsGQ1zxR-jS7sInEh'   # new file
url = f"https://drive.usercontent.google.com/download?id={file_id}&export=download&confirm=t"

df_policy = pd.read_csv(url)
df_policy.info()

<class 'pandas.DataFrame'>
RangeIndex: 391166 entries, 0 to 391165
Data columns (total 72 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   loan_amnt                    391166 non-null  int64  
 1   term                         391166 non-null  int64  
 2   int_rate                     391166 non-null  float64
 3   installment                  391166 non-null  float64
 4   sub_grade                    391166 non-null  str    
 5   emp_length                   391166 non-null  float64
 6   home_ownership               391166 non-null  str    
 7   annual_inc                   391166 non-null  float64
 8   verification_status          391166 non-null  str    
 9   purpose                      391166 non-null  str    
 10  dti                          391166 non-null  float64
 11  delinq_2yrs                  391166 non-null  float64
 12  inq_last_6mths               391166 non-null  float64
 13  open_acc  

### Create Simulated Loan Size Grid

In [6]:
# Loan size multipliers
loan_size_multipliers = [0.50, 0.75, 1.00, 1.25]

# Add borrower ID if needed
df_policy = df_policy.reset_index(drop=False).rename(columns={"index": "borrower_id"})

grid_list = []

for mult in loan_size_multipliers:
    temp = df_policy.copy()
    temp["loan_size_multiplier"] = mult
    temp["sim_loan_amnt"] = (temp["loan_amnt"] * mult).round(2)
    
    # Recalculate key ratios
    temp["sim_income_to_loan_ratio"] = temp["annual_inc"] / temp["sim_loan_amnt"]
    temp["sim_loan_to_income_ratio"] = temp["sim_loan_amnt"] / temp["annual_inc"]
    
    grid_list.append(temp)

loan_grid_df = pd.concat(grid_list, ignore_index=True)

# Clean up infinities
loan_grid_df.replace([np.inf, -np.inf], np.nan, inplace=True)

loan_grid_df.head()

,level_0,loan_amnt,term,int_rate,installment,sub_grade,emp_length,home_ownership,annual_inc,verification_status,...,total_interest,yield_on_principal,expected_lgd,expected_revenue,expected_loss,expected_profit,loan_size_multiplier,sim_loan_amnt,sim_income_to_loan_ratio,sim_loan_to_income_ratio
0,0,2550,36,16.55,90.35,D2,3.0,RENT,72500.0,Source Verified,...,702.40,0.2755,0.655482,599.996798,243.685645,356.311153,0.5,1275.0,56.862745,0.017586
1,1,9600,36,13.99,328.06,C3,1.0,MORTGAGE,73000.0,Source Verified,...,2210.11,0.2302,0.621307,1838.385358,1003.193284,835.192074,0.5,4800.0,15.208333,0.065753
2,2,21000,36,12.35,701.02,B4,2.0,MORTGAGE,78000.0,Source Verified,...,4236.58,0.2017,0.574821,3796.638399,1253.521381,2543.117018,0.5,10500.0,7.428571,0.134615
3,3,6000,36,17.86,216.50,D5,10.0,RENT,50000.0,Verified,...,1793.76,0.2990,0.683216,1302.134949,1123.515798,178.619152,0.5,3000.0,16.666667,0.060000
4,4,20000,60,19.19,520.91,E3,1.0,RENT,40000.0,Verified,...,11254.25,0.5627,0.701859,5477.192799,7205.598633,-1728.405834,0.5,10000.0,4.000000,0.250000
